In [ ]:
file_path_stock  ="/Volumes/workspace/default/tp1_datalake/stocks.csv"
dfstocks= spark.read.format("csv").option("header" , "true").load(file_path_stock)
dfstocks.printSchema()

In [ ]:
dfstocks.createOrReplaceTempView("stocks")

In [ ]:
# doublons en stcoks
import pyspark.sql.functions as F
dfstocks.groupBy("date_stock", "id_magasin", "id_produit") \
    .count() \
    .filter(F.col("count") > 1) \
    .display()

In [ ]:
# Vérifier si on a des champs vides

champs_vide  = dfstocks.filter(
    (F.col("date_stock").isNull()) |
    (F.col("id_magasin").isNull()) |
    (F.col("id_produit").isNull()) |
    (F.col("quantite_stock").isNull())
 
).display()

In [ ]:
%sql
select * from stocks

In [ ]:
%sql
-- recherche de doublons (ok)
SELECT  date_stock,  id_magasin,  id_produit, COUNT(*) AS nombre_occurrences
FROM stocks
GROUP BY date_stock, id_magasin, id_produit
HAVING COUNT(*) > 1
ORDER BY nombre_occurrences DESC;

In [ ]:
%sql
-- rupture de stock (277) rupture de stock (ko)
select id_produit , quantite_stock ,id_magasin  from stocks where quantite_stock = 0
group by id_produit , quantite_stock , id_magasin

In [ ]:
%sql
select * from stocks
where quantite_stock <0

In [ ]:
# référentiels pour vérifier que chaque stock pointe vers un produit et un magasin existants
chemin = "/Volumes/workspace/default/tp1_datalake/"
spark.read.format("csv").option("header", "true").load(chemin + "produits.csv").createOrReplaceTempView("produits_ref")
spark.read.format("csv").option("header", "true").load(chemin + "magasins.csv").createOrReplaceTempView("magasins_ref")

In [ ]:
%sql
-- nombre de lignes : 84 000 (ok)
SELECT COUNT(*) AS nombre_lignes FROM stocks

In [ ]:
%sql
-- ruptures de stock, comptage corrigé : 277 lignes à 0 (271 couples produit/magasin, certains en rupture 2 mois)
-- ce n'est pas une erreur de données mais une réalité métier (ok : garder et signaler, utile pour les prévisions de Léa)
SELECT COUNT(*) AS lignes_en_rupture,
       COUNT(DISTINCT id_produit, id_magasin) AS couples_en_rupture
FROM stocks
WHERE CAST(quantite_stock AS INT) = 0

In [ ]:
%sql
-- types : date_stock et quantite_stock sont lus en texte (ok : tout se convertit en date et en entier)
SELECT SUM(CASE WHEN try_cast(date_stock AS DATE) IS NULL THEN 1 ELSE 0 END)    AS dates_illisibles,
       SUM(CASE WHEN try_cast(quantite_stock AS INT) IS NULL THEN 1 ELSE 0 END) AS quantites_illisibles
FROM stocks

In [ ]:
%sql
-- min / max (ok : stock de 0 à 299, du 31/01/2025 au 30/09/2026)
SELECT MIN(CAST(quantite_stock AS INT)) AS stock_min,
       MAX(CAST(quantite_stock AS INT)) AS stock_max,
       MIN(to_date(date_stock))         AS premiere_date,
       MAX(to_date(date_stock))         AS derniere_date
FROM stocks

In [ ]:
%sql
-- valeurs distinctes et grille complète : 21 mois x 20 magasins x 200 produits = 84 000 lignes (ok : aucune combinaison manquante)
SELECT COUNT(DISTINCT date_stock) AS mois,
       COUNT(DISTINCT id_magasin) AS magasins,
       COUNT(DISTINCT id_produit) AS produits,
       COUNT(DISTINCT date_stock) * COUNT(DISTINCT id_magasin) * COUNT(DISTINCT id_produit) AS combinaisons_attendues,
       COUNT(*) AS lignes
FROM stocks

In [ ]:
%sql
-- produits ou magasins inconnus (ok)
SELECT
  (SELECT COUNT(*) FROM stocks s LEFT ANTI JOIN produits_ref p ON s.id_produit = p.id_produit) AS produits_inconnus,
  (SELECT COUNT(*) FROM stocks s LEFT ANTI JOIN magasins_ref m ON s.id_magasin = m.id_magasin) AS magasins_inconnus

### Stocks : interpréter, décider, justifier

| Constat | Interprétation | Décision |
|---|---|---|
| 84 000 lignes, aucun doublon sur (date, magasin, produit), aucun vide | Une photo par fin de mois, grille complète (21 × 20 × 200) | ok |
| Tout est lu en texte | Erreur technique de lecture : le CSV n'a pas de types | **Corriger** : convertir date_stock en date et quantite_stock en entier |
| Aucun stock négatif (de 0 à 299) | Valeurs plausibles | ok. Un stock négatif est physiquement impossible : **quarantaine**, sans le remplacer par 0 |
| 277 lignes à 0 (271 couples produit/magasin) | **Réalité métier** : ce sont des ruptures, pas des erreurs | **Garder et signaler** : c'est exactement ce que Léa veut prévoir |
| Aucun produit ni magasin inconnu | Intégrité respectée | ok |

**Pour Léa** : l'historique complet, sans valeur inventée, permet de prévoir les ruptures produit par produit.

## Passage en silver : on applique les décisions
1. **Corriger** : convertir date_stock en date et quantite_stock en entier.
2. **Mettre en quarantaine** : date illisible, stock vide ou négatif, doublon sur (date, magasin, produit). Un stock négatif n'est **pas** remplacé par 0 : cela inventerait des ruptures.
3. **Garder et signaler** : les ruptures (stock = 0) avec la colonne est_rupture.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# 1. Convertir les types (le CSV est lu en texte)
stocks_types = dfstocks \
    .withColumn("date_stock", F.expr("try_cast(date_stock AS DATE)")) \
    .withColumn("quantite_stock", F.expr("try_cast(quantite_stock AS INT)"))

# 2. Règles de quarantaine
stocks_controle = stocks_types \
    .withColumn("nb_cle", F.count("*").over(Window.partitionBy("date_stock", "id_magasin", "id_produit"))) \
    .withColumn("motif_rejet",
        F.when(F.col("date_stock").isNull(), "date illisible")
         .when(F.col("quantite_stock").isNull(), "stock vide ou illisible")
         .when(F.col("quantite_stock") < 0, "stock negatif")
         .when(F.col("nb_cle") > 1, "doublon sur la cle")) \
    .drop("nb_cle")

# 3. Garder et signaler : les ruptures de stock
stocks_controle = stocks_controle.withColumn("est_rupture", F.col("quantite_stock") == 0)

stocks_quarantaine = stocks_controle.filter(F.col("motif_rejet").isNotNull())
stocks_silver = stocks_controle.filter(F.col("motif_rejet").isNull()).drop("motif_rejet")
print("Lignes en silver :", stocks_silver.count())                 # 84 000
print("Lignes en quarantaine :", stocks_quarantaine.count())       # 0
print("Ruptures signalées :", stocks_silver.filter("est_rupture").count())   # 277
display(stocks_silver.limit(10))

In [ ]:
# Conserver la table stocks nettoyée dans la couche Silver
stocks_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/tp1_datalake/silver/stocks")

# Les lignes rejetées sont conservées à part, avec leur motif (quarantaine)
stocks_quarantaine.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/tp1_datalake/silver/quarantaine_stocks")

In [ ]:
# Vérification de la table silver
verif = spark.read.format("delta").load("/Volumes/workspace/default/tp1_datalake/silver/stocks")
verif.printSchema()
print("Lignes en silver :", verif.count())   # 84 000